In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import random
from datetime import datetime, timedelta

In [ ]:
%reload_ext watermark


In [ ]:
watermark - -iversions

In [ ]:
def gerar_dados_ficticios(num_registros=600):
    produtos = {
        "Laptop Gamer": {"categoria": "Eletrônicos", "preco": 7500.00},
        "Mouse Vertical": {"categoria": "Acessórios", "preco": 250.00},
        "Teclado Mecânico": {"categoria": "Acessórios", "preco": 550.00},
        "Monitor Ultrawide": {"categoria": "Eletrônicos", "preco": 2800.00},
        "Cadeira Gamer": {"categoria": "Móveis", "preco": 1200.00},
        "Headset 7.1": {"categoria": "Acessórios", "preco": 800.00},
        "Placa de Vídeo": {"categoria": "Hardware", "preco": 4500.00},
        "SSD 1TB": {"categoria": "Hardware", "preco": 600.00},
    }
    lista_produtos = list(produtos.keys())
    cidades_estados = {
        "São Paulo": "SP",
        "Rio de Janeiro": "RJ",
        "Belo Horizonte": "MG",
        "Porto Alegre": "RS",
        "Salvador": "BA",
        "Curitiba": "PR",
        "Fortaleza": "CE",
    }
    lista_cidades = list(cidades_estados.keys())

    dados_vendas = []
    data_inicial = datetime(2026, 1, 1)

    for i in range(num_registros):
        produto_nome = random.choice(lista_produtos)

        cidade = random.choice(lista_cidades)

        quantidade = np.random.randint(1, 8)

        data_pedido = data_inicial + timedelta(
            days=int(i / 5), hours=random.randint(0, 23)
        )

        if produto_nome in ["Mouse Vertical", "Teclado Mecânico"]:
            preco_unitario = produtos[produto_nome]["preco"] * np.random.uniform(
                0.9, 1.0
            )
        else:
            preco_unitario = produtos[produto_nome]["preco"]

        dados_vendas.append(
            {
                "ID_Pedido": 1000 + i,
                "Data_Pedido": data_pedido,
                "Nome_Produto": produto_nome,
                "Categoria": produtos[produto_nome]["categoria"],
                "Preco_Unitario": round(preco_unitario, 2),
                "Quantidade": quantidade,
                "ID_Cliente": np.random.randint(100, 150),
                "Cidade": cidade,
                "Estado": cidades_estados[cidade],
            }
        )

    return pd.DataFrame(dados_vendas)

In [ ]:
df_vendas = gerar_dados_ficticios(500)

In [ ]:
df_vendas.shape

In [ ]:
df_vendas.head()

In [ ]:
df_vendas.info()

In [ ]:
df_vendas.describe()

In [ ]:
df_vendas["Data_Pedido"] = pd.to_datetime(df_vendas["Data_Pedido"])

In [ ]:
df_vendas["Faturamento"] = df_vendas["Preco_Unitario"] * df_vendas["Quantidade"]

In [ ]:
df_vendas["Status_Entrega"] = df_vendas["Estado"].apply(
    lambda estado: "Rapida" if estado in ["SP", "RJ", "MJ"] else "Normal"
)

In [ ]:
df_vendas.info()

In [ ]:
df_vendas.head()

## Análise 1 - Top 10 Produtos Mais Vendidos

In [ ]:
top_10_produtos = (
    df_vendas.groupby("Nome_Produto")["Quantidade"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

In [ ]:
top_10_produtos

In [ ]:
sns.set_style("whitegrid")

plt.Figure(figsize=(12, 7))

top_10_produtos.sort_values(ascending=True).plot(kind="barh", color="skyblue")

plt.title("Top 10 Produtos mais vendidos", fontsize=16)
plt.xlabel("Quantidade Vendida", fontsize=12)
plt.ylabel("Produtos", fontsize=12)

plt.tight_layout()
plt.show()

## Análise 2 - Faturamento Mensal

In [ ]:
df_vendas["Mes"] = df_vendas["Data_Pedido"].dt.to_period("M")

In [ ]:
faturamento_mensal = df_vendas.groupby("Mes")["Faturamento"].sum()

In [ ]:
df_vendas.head()

In [ ]:
faturamento_mensal.index = faturamento_mensal.index.strftime("%Y-%m")

In [ ]:
faturamento_mensal.map("R$ {:,.2f}".format)

In [ ]:
plt.Figure(figsize=(12, 6))

faturamento_mensal.plot(kind="line", marker="o", linestyle="-", color="green")

plt.title("Evolução do Faturamento Mensal", fontsize=16)
plt.xlabel("Mês", fontsize=12)

plt.ylabel("Faturamento (R$)", fontsize=12)

plt.grid(True, which="both", linestyle="--", linewidth=0.5)

plt.tight_layout()

plt.show()

## Análise 3 - Vendas Por Estado

In [ ]:
vendas_estado = (
    df_vendas.groupby("Estado")["Faturamento"].sum().sort_values(ascending=False)
)

In [ ]:
vendas_estado.map("R$ {:,.2f}".format)

In [ ]:
plt.Figure(figsize=(12, 7))

vendas_estado.plot(kind="bar", color=sns.color_palette("husl", 7))

plt.title("Faturamento Por Estado", fontsize=16)
plt.xlabel("Estado", fontsize=12)

plt.ylabel("Faturamento (R$)", fontsize=12)

plt.xticks(rotation=0)

plt.tight_layout()

plt.show()

## Análise 4 - Faturamento Por Categoria

In [ ]:
faturamento_categoria = (
    df_vendas.groupby("Categoria")["Faturamento"].sum().sort_values(ascending=False)
)

In [ ]:
faturamento_categoria.map("R$ {:,.2f}".format)

In [ ]:
from matplotlib.ticker import FuncFormatter

faturamento_ordenado = faturamento_categoria.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))


def formatar_milhares(y, pos):
    return f"R$ {y/1000:,.0f}K"


formater = FuncFormatter(formatar_milhares)

ax.yaxis.set_major_formatter(formater)

faturamento_ordenado.plot(
    kind="bar", ax=ax, color=sns.color_palette("viridis", len(faturamento_ordenado))
)

ax.set_title("Faturamento Por Categoria", fontsize=16)
ax.set_xlabel("Categoria", fontsize=12)
ax.set_ylabel("Faturamento", fontsize=12)

plt.xticks(rotation=45, ha="right")

plt.tight_layout()

plt.show()